# R-402 Mechanical Design

In [1]:
import math
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 1. Design Inputs

In [2]:
N_tubes = 1300
tube_ID_mm = 40.0
tube_OD_mm = 44.0
packed_length_m = 6.50
catalyst_mass_kg = 9420.0
tube_pitch_mm = 55.0
bundle_diameter_m = 2.132
shell_ID_m = 2.25

reactor_inlet_T_C = 223.0
max_reactor_T_C = 249.7
cooling_water_T_C = 225.0
heat_removed_kW = 7373.96

reactor_mass_flow_kg_h = 26781.209
rho_inlet_gas_kg_m3 = 12.76   # Stream 405
rho_outlet_gas_kg_m3 = 16.46  # Stream 406

tube_side_design_pressure_barg = 55.0
shell_side_design_pressure_barg = 30.0
design_temperature_C = 280.0

selected_shell_thickness_mm = 36.0
selected_head_thickness_mm = 60.0
selected_channel_thickness_mm = 60.0
selected_tube_thickness_mm = 2.0

corrosion_allowance_mm = 1.0
joint_efficiency_welded = 0.85
joint_efficiency_seamless_tube = 1.00

S_SA516_Gr70_MPa = 136.0
S_SA387_Gr22Cl2_MPa = 170.0
S_SA213_T22_MPa = 103.4

rho_steel_kg_m3 = 7850.0

gas_channel_height_m = 0.50
skirt_height_m = 1.50
normal_shell_water_fill_fraction = 0.70
misc_weight_allowance_fraction = 0.20  # tubesheets, supports, nozzles, small internals, etc.

process_gas_nozzle_velocity_m_s = 20.0
steam_nozzle_velocity_m_s = 25.0
BFW_nozzle_velocity_m_s = 2.0

standard_DN_mm = np.array([25, 32, 40, 50, 65, 80, 100, 125, 150, 200, 250, 300, 350, 400])

steam_sat_pressure_MPa_abs = 2.5497
rho_sat_liquid_kg_m3 = 833.75
rho_sat_vapour_kg_m3 = 12.755
h_sat_liquid_kJ_kg = 966.80
h_sat_vapour_kJ_kg = 2802.1
h_fg_kJ_kg = 1835.4

print("Inputs loaded.")

Inputs loaded.


## 2. Tube Bundle and Shell Geometry

In [3]:
tube_pitch_calc_mm = 1.25 * tube_OD_mm
pitch_error_mm = tube_pitch_mm - tube_pitch_calc_mm
clear_ligament_mm = tube_pitch_mm - tube_OD_mm

bundle_to_shell_total_clearance_m = shell_ID_m - bundle_diameter_m
bundle_to_shell_radial_clearance_mm = 1000 * bundle_to_shell_total_clearance_m / 2

shell_OD_m = shell_ID_m + 2 * selected_shell_thickness_mm / 1000

geometry_summary = pd.DataFrame({
    "Parameter": [
        "Number of catalyst tubes",
        "Tube ID",
        "Tube OD",
        "Calculated 1.25 d_o pitch",
        "Selected tube pitch",
        "Clear ligament between tubes",
        "Tube bundle diameter",
        "Shell internal diameter",
        "Radial bundle-to-shell clearance",
        "Selected shell thickness",
        "Shell outside diameter",
        "Packed catalyst length"
    ],
    "Value": [
        N_tubes,
        tube_ID_mm,
        tube_OD_mm,
        tube_pitch_calc_mm,
        tube_pitch_mm,
        clear_ligament_mm,
        bundle_diameter_m,
        shell_ID_m,
        bundle_to_shell_radial_clearance_mm,
        selected_shell_thickness_mm,
        shell_OD_m,
        packed_length_m
    ],
    "Unit": [
        "-", "mm", "mm", "mm", "mm", "mm",
        "m", "m", "mm", "mm", "m", "m"
    ]
})

geometry_summary

,Parameter,Value,Unit
0,Number of catalyst tubes,1300.000,-
1,Tube ID,40.000,mm
2,Tube OD,44.000,mm
3,Calculated 1.25 d_o pitch,55.000,mm
4,Selected tube pitch,55.000,mm
5,Clear ligament between tubes,11.000,mm
6,Tube bundle diameter,2.132,m
7,Shell internal diameter,2.250,m
8,Radial bundle-to-shell clearance,59.000,mm
9,Selected shell thickness,36.000,mm


## 3. Pressure Thickness Calculations

In [4]:
def cylindrical_required_thickness_mm(P_MPa, R_mm, S_MPa, E):
    denom = S_MPa * E - 0.6 * P_MPa
    if denom <= 0:
        raise ValueError("Invalid inputs: denominator is non-positive.")
    return P_MPa * R_mm / denom

def ellipsoidal_head_required_thickness_mm(P_MPa, D_mm, S_MPa, E):
    denom = 2.0 * S_MPa * E - 0.2 * P_MPa
    if denom <= 0:
        raise ValueError("Invalid inputs: denominator is non-positive.")
    return P_MPa * D_mm / denom

P_shell_MPa = shell_side_design_pressure_barg * 0.1
P_tube_MPa = tube_side_design_pressure_barg * 0.1
R_shell_mm = shell_ID_m * 1000 / 2
D_shell_mm = shell_ID_m * 1000
R_tube_mm = tube_ID_mm / 2

t_shell_calc_mm = cylindrical_required_thickness_mm(
    P_shell_MPa, R_shell_mm, S_SA516_Gr70_MPa, joint_efficiency_welded
)
t_shell_with_CA_mm = t_shell_calc_mm + corrosion_allowance_mm

t_head_calc_mm = ellipsoidal_head_required_thickness_mm(
    P_tube_MPa, D_shell_mm, S_SA387_Gr22Cl2_MPa, joint_efficiency_welded
)
t_head_with_CA_mm = t_head_calc_mm + corrosion_allowance_mm

t_channel_calc_mm = cylindrical_required_thickness_mm(
    P_tube_MPa, R_shell_mm, S_SA387_Gr22Cl2_MPa, joint_efficiency_welded
)
t_channel_with_CA_mm = t_channel_calc_mm + corrosion_allowance_mm

t_tube_calc_mm = cylindrical_required_thickness_mm(
    P_tube_MPa, R_tube_mm, S_SA213_T22_MPa, joint_efficiency_seamless_tube
)

thickness_checks = pd.DataFrame({
    "Component": [
        "Main shell",
        "Gas-side 2:1 ellipsoidal head",
        "Gas-side cylindrical channel",
        "Catalyst tube"
    ],
    "Material": [
        "SA-516 Gr. 70",
        "SA-387 Gr. 22 Cl. 2",
        "SA-387 Gr. 22 Cl. 2",
        "SA-213 T22"
    ],
    "Design pressure (bar(g))": [
        shell_side_design_pressure_barg,
        tube_side_design_pressure_barg,
        tube_side_design_pressure_barg,
        tube_side_design_pressure_barg
    ],
    "Allowable stress input (MPa)": [
        S_SA516_Gr70_MPa,
        S_SA387_Gr22Cl2_MPa,
        S_SA387_Gr22Cl2_MPa,
        S_SA213_T22_MPa
    ],
    "Calculated pressure thickness (mm)": [
        t_shell_calc_mm,
        t_head_calc_mm,
        t_channel_calc_mm,
        t_tube_calc_mm
    ],
    "Calculated + CA (mm)": [
        t_shell_with_CA_mm,
        t_head_with_CA_mm,
        t_channel_with_CA_mm,
        t_tube_calc_mm
    ],
    "Selected thickness (mm)": [
        selected_shell_thickness_mm,
        selected_head_thickness_mm,
        selected_channel_thickness_mm,
        selected_tube_thickness_mm
    ]
})

thickness_checks["Margin to selected (mm)"] = (
    thickness_checks["Selected thickness (mm)"] - thickness_checks["Calculated + CA (mm)"]
)
thickness_checks["Pass preliminary check"] = thickness_checks["Margin to selected (mm)"] >= 0

thickness_checks.round(3)

,Component,Material,Design pressure (bar(g)),Allowable stress input (MPa),Calculated pressure thickness (mm),Calculated + CA (mm),Selected thickness (mm),Margin to selected (mm),Pass preliminary check
0,Main shell,SA-516 Gr. 70,30.0,136.0,29.657,30.657,36.0,5.343,True
1,Gas-side 2:1 ellipsoidal head,SA-387 Gr. 22 Cl. 2,55.0,170.0,42.984,43.984,60.0,16.016,True
2,Gas-side cylindrical channel,SA-387 Gr. 22 Cl. 2,55.0,170.0,43.821,44.821,60.0,15.179,True
3,Catalyst tube,SA-213 T22,55.0,103.4,1.099,1.099,2.0,0.901,True


## 4. Head Geometry and Overall Height

In [5]:
head_internal_depth_m = shell_ID_m / 4

pressure_vessel_height_m = (
    packed_length_m
    + 2 * gas_channel_height_m
    + 2 * head_internal_depth_m
)

installed_height_m = pressure_vessel_height_m + skirt_height_m

height_summary = pd.DataFrame({
    "Dimension": [
        "Packed / active tube length",
        "Upper gas channel allowance",
        "Lower gas channel allowance",
        "2:1 head internal depth (each)",
        "Pressure-vessel height excluding skirt",
        "Skirt height allowance",
        "Preliminary installed height excluding nozzle projection"
    ],
    "Value (m)": [
        packed_length_m,
        gas_channel_height_m,
        gas_channel_height_m,
        head_internal_depth_m,
        pressure_vessel_height_m,
        skirt_height_m,
        installed_height_m
    ]
})

height_summary.round(4)

,Dimension,Value (m)
0,Packed / active tube length,6.5000
1,Upper gas channel allowance,0.5000
2,Lower gas channel allowance,0.5000
3,2:1 head internal depth (each),0.5625
4,Pressure-vessel height excluding skirt,8.6250
5,Skirt height allowance,1.5000
6,Preliminary installed height excluding nozzle ...,10.1250


## 5. Steam Generation and BFW Requirement

In [6]:
steam_generation_kg_s = heat_removed_kW / h_fg_kJ_kg
steam_generation_kg_h = steam_generation_kg_s * 3600

steam_results = pd.DataFrame({
    "Parameter": [
        "Boiling-water temperature",
        "Saturation pressure",
        "Heat removed from R-402",
        "Saturated-liquid enthalpy",
        "Saturated-vapour enthalpy",
        "Latent heat at 225 °C",
        "Net steam generation"
    ],
    "Value": [
        cooling_water_T_C,
        steam_sat_pressure_MPa_abs,
        heat_removed_kW,
        h_sat_liquid_kJ_kg,
        h_sat_vapour_kJ_kg,
        h_fg_kJ_kg,
        steam_generation_kg_h
    ],
    "Unit": [
        "°C", "MPa(abs)", "kW", "kJ/kg", "kJ/kg", "kJ/kg", "kg/h"
    ]
})

steam_results.round(3)

,Parameter,Value,Unit
0,Boiling-water temperature,225.000,°C
1,Saturation pressure,2.550,MPa(abs)
2,Heat removed from R-402,7373.960,kW
3,Saturated-liquid enthalpy,966.800,kJ/kg
4,Saturated-vapour enthalpy,2802.100,kJ/kg
5,Latent heat at 225 °C,1835.400,kJ/kg
6,Net steam generation,14463.472,kg/h


## 6. Preliminary Nozzle Sizing

In [7]:
def required_diameter_m(mass_flow_kg_s, density_kg_m3, velocity_m_s):
    volumetric_flow_m3_s = mass_flow_kg_s / density_kg_m3
    area_m2 = volumetric_flow_m3_s / velocity_m_s
    diameter_m = math.sqrt(4 * area_m2 / math.pi)
    return volumetric_flow_m3_s, area_m2, diameter_m

def next_DN(required_diameter_m):
    required_mm = required_diameter_m * 1000
    larger = standard_DN_mm[standard_DN_mm >= required_mm]
    return int(larger[0]) if len(larger) else int(math.ceil(required_mm / 50) * 50)

process_mass_flow_kg_s = reactor_mass_flow_kg_h / 3600

Qv_in, A_in, D_in = required_diameter_m(
    process_mass_flow_kg_s, rho_inlet_gas_kg_m3, process_gas_nozzle_velocity_m_s
)

Qv_out, A_out, D_out = required_diameter_m(
    process_mass_flow_kg_s, rho_outlet_gas_kg_m3, process_gas_nozzle_velocity_m_s
)

Qv_steam, A_steam, D_steam = required_diameter_m(
    steam_generation_kg_s, rho_sat_vapour_kg_m3, steam_nozzle_velocity_m_s
)

Qv_bfw, A_bfw, D_bfw = required_diameter_m(
    steam_generation_kg_s, rho_sat_liquid_kg_m3, BFW_nozzle_velocity_m_s
)

nozzle_data = [
    ["N1", "Synthesis-gas inlet", process_mass_flow_kg_s, rho_inlet_gas_kg_m3,
     process_gas_nozzle_velocity_m_s, D_in*1000, next_DN(D_in)],
    ["N2", "Reactor-effluent outlet", process_mass_flow_kg_s, rho_outlet_gas_kg_m3,
     process_gas_nozzle_velocity_m_s, D_out*1000, next_DN(D_out)],
    ["N3", "Generated-steam outlet", steam_generation_kg_s, rho_sat_vapour_kg_m3,
     steam_nozzle_velocity_m_s, D_steam*1000, next_DN(D_steam)],
    ["N4", "Preheated BFW inlet", steam_generation_kg_s, rho_sat_liquid_kg_m3,
     BFW_nozzle_velocity_m_s, D_bfw*1000, next_DN(D_bfw)],
]

nozzle_summary = pd.DataFrame(nozzle_data, columns=[
    "Nozzle", "Service", "Mass flow (kg/s)", "Density (kg/m³)",
    "Target velocity (m/s)", "Calculated ID (mm)", "Preliminary DN"
])

nozzle_summary.round(2)

,Nozzle,Service,Mass flow (kg/s),Density (kg/m³),Target velocity (m/s),Calculated ID (mm),Preliminary DN
0,N1,Synthesis-gas inlet,7.44,12.76,20.0,192.65,200
1,N2,Reactor-effluent outlet,7.44,16.46,20.0,169.62,200
2,N3,Generated-steam outlet,4.02,12.76,25.0,126.66,150
3,N4,Preheated BFW inlet,4.02,833.75,2.0,55.39,65


## 7. Preliminary Reactor Weight

In [8]:
shell_t_m = selected_shell_thickness_mm / 1000
shell_OD_m = shell_ID_m + 2 * shell_t_m
V_shell_steel_m3 = math.pi/4 * (shell_OD_m**2 - shell_ID_m**2) * packed_length_m
m_shell_kg = V_shell_steel_m3 * rho_steel_kg_m3

tube_ID_m = tube_ID_mm / 1000
tube_OD_m = tube_OD_mm / 1000
V_tube_steel_m3 = N_tubes * math.pi/4 * (tube_OD_m**2 - tube_ID_m**2) * packed_length_m
m_tubes_kg = V_tube_steel_m3 * rho_steel_kg_m3

a = shell_ID_m / 2
c = shell_ID_m / 4
ecc = math.sqrt(1 - (c/a)**2)
A_full_oblate_m2 = 2 * math.pi * a**2 * (
    1 + ((1 - ecc**2)/ecc) * 0.5 * math.log((1 + ecc)/(1 - ecc))
)
A_one_head_m2 = A_full_oblate_m2 / 2

head_t_m = selected_head_thickness_mm / 1000
m_two_heads_kg = 2 * A_one_head_m2 * head_t_m * rho_steel_kg_m3

channel_t_m = selected_channel_thickness_mm / 1000
channel_OD_m = shell_ID_m + 2 * channel_t_m
V_channels_steel_m3 = 2 * (
    math.pi/4 * (channel_OD_m**2 - shell_ID_m**2) * gas_channel_height_m
)
m_channels_kg = V_channels_steel_m3 * rho_steel_kg_m3

shell_internal_cross_section_m2 = math.pi/4 * shell_ID_m**2
tube_external_area_total_m2 = N_tubes * math.pi/4 * tube_OD_m**2
shell_side_free_volume_m3 = (
    shell_internal_cross_section_m2 - tube_external_area_total_m2
) * packed_length_m

m_shell_water_kg = (
    shell_side_free_volume_m3
    * normal_shell_water_fill_fraction
    * rho_sat_liquid_kg_m3
)

known_operating_subtotal_kg = (
    m_shell_kg + m_tubes_kg + m_two_heads_kg + m_channels_kg
    + catalyst_mass_kg + m_shell_water_kg
)

prelim_operating_mass_kg = known_operating_subtotal_kg * (1 + misc_weight_allowance_fraction)

weight_summary = pd.DataFrame({
    "Item": [
        "Cylindrical shell steel",
        "Catalyst tubes",
        "Two gas-side head skins",
        "Two gas-side channel shells",
        "Catalyst",
        f"Shell-side water ({normal_shell_water_fill_fraction:.0%} fill)",
        "Known subtotal",
        f"Preliminary operating mass incl. {misc_weight_allowance_fraction:.0%} misc. allowance"
    ],
    "Mass (kg)": [
        m_shell_kg,
        m_tubes_kg,
        m_two_heads_kg,
        m_channels_kg,
        catalyst_mass_kg,
        m_shell_water_kg,
        known_operating_subtotal_kg,
        prelim_operating_mass_kg
    ]
})

weight_summary.round(0)

,Item,Mass (kg)
0,Cylindrical shell steel,13192.0
1,Catalyst tubes,17505.0
2,Two gas-side head skins,5169.0
3,Two gas-side channel shells,3418.0
4,Catalyst,9420.0
5,Shell-side water (70% fill),7585.0
6,Known subtotal,56289.0
7,Preliminary operating mass incl. 20% misc. all...,67547.0


## 8. Final Mechanical Design Results

In [9]:
final_results = pd.DataFrame([
    ["Number of catalyst tubes", N_tubes, "-"],
    ["Tube ID / OD", f"{tube_ID_mm:.0f} / {tube_OD_mm:.0f}", "mm"],
    ["Tube pitch", tube_pitch_mm, "mm"],
    ["Packed catalyst length", packed_length_m, "m"],
    ["Catalyst inventory", catalyst_mass_kg/1000, "t"],
    ["Tube bundle diameter", bundle_diameter_m, "m"],
    ["Shell internal diameter", shell_ID_m, "m"],
    ["Shell outside diameter", shell_OD_m, "m"],
    ["Radial bundle-to-shell clearance", bundle_to_shell_radial_clearance_mm, "mm"],
    ["Shell-side operating temperature", cooling_water_T_C, "°C"],
    ["Shell-side saturation pressure at 225 °C", steam_sat_pressure_MPa_abs, "MPa(abs)"],
    ["Tube-side design pressure", tube_side_design_pressure_barg, "bar(g)"],
    ["Shell-side design pressure", shell_side_design_pressure_barg, "bar(g)"],
    ["Mechanical design temperature", design_temperature_C, "°C"],
    ["Calculated shell thickness + CA", t_shell_with_CA_mm, "mm"],
    ["Selected shell thickness", selected_shell_thickness_mm, "mm"],
    ["Calculated gas-head thickness + CA", t_head_with_CA_mm, "mm"],
    ["Selected gas-head thickness", selected_head_thickness_mm, "mm"],
    ["Calculated gas-channel thickness + CA", t_channel_with_CA_mm, "mm"],
    ["Selected gas-channel thickness", selected_channel_thickness_mm, "mm"],
    ["Calculated catalyst-tube minimum thickness", t_tube_calc_mm, "mm"],
    ["Selected catalyst-tube wall thickness", selected_tube_thickness_mm, "mm"],
    ["2:1 ellipsoidal-head internal depth", head_internal_depth_m, "m"],
    ["Preliminary pressure-vessel height excl. skirt", pressure_vessel_height_m, "m"],
    ["Preliminary skirt height", skirt_height_m, "m"],
    ["Preliminary installed height excl. nozzle projection", installed_height_m, "m"],
    ["Net steam generation at 225 °C", steam_generation_kg_h, "kg/h"],
    ["N1 preliminary synthesis-gas inlet", f"DN {next_DN(D_in)}", "-"],
    ["N2 preliminary reactor-effluent outlet", f"DN {next_DN(D_out)}", "-"],
    ["N3 preliminary steam outlet", f"DN {next_DN(D_steam)}", "-"],
    ["N4 preliminary BFW inlet", f"DN {next_DN(D_bfw)}", "-"],
    ["Preliminary operating mass estimate", prelim_operating_mass_kg/1000, "t"],
], columns=["Mechanical design result", "Value", "Unit"])

final_results

,Mechanical design result,Value,Unit
0,Number of catalyst tubes,1300,-
1,Tube ID / OD,40 / 44,mm
2,Tube pitch,55.0,mm
3,Packed catalyst length,6.5,m
4,Catalyst inventory,9.42,t
5,Tube bundle diameter,2.132,m
6,Shell internal diameter,2.25,m
7,Shell outside diameter,2.322,m
8,Radial bundle-to-shell clearance,59.0,mm
9,Shell-side operating temperature,225.0,°C


In [10]:
geometry_summary.to_csv("/mnt/data/R402_Chapter4_Geometry.csv", index=False)
thickness_checks.to_csv("/mnt/data/R402_Chapter4_Thickness_Checks.csv", index=False)
nozzle_summary.to_csv("/mnt/data/R402_Chapter4_Nozzles.csv", index=False)
weight_summary.to_csv("/mnt/data/R402_Chapter4_Weight_Estimate.csv", index=False)
final_results.to_csv("/mnt/data/R402_Chapter4_Final_Results.csv", index=False)

print("CSV result tables exported to /mnt/data.")

CSV result tables exported to /mnt/data.


## 9. Detailed Design Exclusions